# AI-Assisted Medical Imaging — ISIC 2016 Task 3B

**Environment:** Google Colab  
**Dataset:** ISIC 2016 Task 3B — 900 dermoscopic images, 900 expert segmentation masks, and benign/malignant labels.

This notebook downloads the training data, validates image-mask pairs, builds a Polars metadata table, explores the data, creates stratified train/validation/test splits, and trains two compact baseline models:
1. ResNet18 for benign/malignant classification.
2. A small U-Net for lesion segmentation.

> Research/education use only. This notebook is not a clinical diagnostic system. The dataset is small, so results must be interpreted cautiously.

## 1. Dataset and source

Official challenge description:  
https://challenge.isic-archive.com/landing/2016/41/

The official page describes 900 JPEG images, 900 matching PNG masks, and a 900-row CSV with benign/malignant labels. Masks use 0 for background and 255 for lesion.

The download links below point to the ISIC challenge's S3 archive. If a link returns an error, open the official challenge page above and use its **Download Training Data** and **Download Training Ground Truth** links.

In [ ]:
# Install lightweight data/visualization dependencies.
!pip -q install polars scikit-learn pillow matplotlib tqdm

In [ ]:
import os
import zipfile
import shutil
import random
import requests
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path("/content/ai_medical_imaging_isic")
RAW_DIR = PROJECT_DIR / "data" / "raw"
IMAGE_DIR = PROJECT_DIR / "data" / "images"
MASK_DIR = PROJECT_DIR / "data" / "masks"
TABLE_DIR = PROJECT_DIR / "data" / "metadata"
OUTPUT_DIR = PROJECT_DIR / "outputs"
MODEL_DIR = OUTPUT_DIR / "models"

for p in [RAW_DIR, IMAGE_DIR, MASK_DIR, TABLE_DIR, OUTPUT_DIR, MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)

## 2. Download the ISIC training files

The image archive contains both JPEG images and corresponding segmentation masks. The CSV contains the classification ground truth.

The download helper streams data to disk rather than loading the full archive into RAM.

In [ ]:
FILES = {
    "training_data.zip": (
        "https://isic-challenge-data.s3.amazonaws.com/2016/"
        "ISBI2016_ISIC_Part3B_Training_Data.zip"
    ),
    "training_labels.csv": (
        "https://isic-challenge-data.s3.amazonaws.com/2016/"
        "ISBI2016_ISIC_Part3B_Training_GroundTruth.csv"
    ),
}

def download_file(url: str, destination: Path, chunk_size: int = 1024 * 1024):
    if destination.exists() and destination.stat().st_size > 0:
        print(f"Already present: {destination.name} ({destination.stat().st_size:,} bytes)")
        return

    print(f"Downloading {destination.name} ...")
    response = requests.get(url, stream=True, timeout=(30, 180))
    response.raise_for_status()

    total = int(response.headers.get("content-length", 0))
    temp_path = destination.with_suffix(destination.suffix + ".part")

    with open(temp_path, "wb") as f, tqdm(
        total=total if total else None,
        unit="B",
        unit_scale=True,
        desc=destination.name
    ) as progress:
        for chunk in response.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)
                progress.update(len(chunk))

    temp_path.replace(destination)
    print(f"Saved: {destination}")

for filename, url in FILES.items():
    download_file(url, RAW_DIR / filename)

In [ ]:
# Check the downloaded files before extraction.
for p in sorted(RAW_DIR.iterdir()):
    print(p.name, f"{p.stat().st_size / (1024**2):.1f} MB")

zip_path = RAW_DIR / "training_data.zip"
assert zipfile.is_zipfile(zip_path), (
    "The downloaded training archive is not a valid ZIP. "
    "Open the official ISIC Task 3B page and download its Training Data ZIP."
)

## 3. Extract and locate images/masks

The code searches recursively because ZIP files can contain nested folders. It does not duplicate image files.

In [ ]:
with zipfile.ZipFile(zip_path, "r") as zf:
    bad_member = zf.testzip()
    if bad_member is not None:
        raise RuntimeError(f"Corrupt ZIP member: {bad_member}")
    zf.extractall(IMAGE_DIR)

image_paths = sorted(IMAGE_DIR.rglob("ISIC_*.jpg"))
mask_paths = sorted(IMAGE_DIR.rglob("ISIC_*_Segmentation.png"))

print("JPEG images:", len(image_paths))
print("PNG masks:", len(mask_paths))

if not image_paths or not mask_paths:
    raise FileNotFoundError(
        "Could not find expected ISIC JPEGs and segmentation PNGs. "
        "Inspect the extracted archive contents and confirm the Task 3B archive was downloaded."
    )

## 4. Read classification labels with Polars

The official challenge defines two fields: image ID (`ISIC_...`) and diagnosis (`benign` or `malignant`). This parser handles either a headerless CSV or a CSV with a header, and checks the detected columns before continuing.

In [ ]:
label_file = RAW_DIR / "training_labels.csv"

# Read as strings so image IDs and labels are not accidentally type-inferred.
labels_raw = pl.read_csv(
    label_file,
    has_header=False,
    infer_schema=False,
    ignore_errors=False
)

print("Raw labels shape:", labels_raw.shape)
display(labels_raw.head(5))

In [ ]:
# Identify the ID and diagnosis columns by their actual values.
id_col = None
diagnosis_col = None

for col in labels_raw.columns:
    values = labels_raw[col].drop_nulls().to_list()
    normalized = [str(v).strip() for v in values]
    if any(v.startswith("ISIC_") for v in normalized):
        id_col = col
    if any(v.lower() in {"benign", "malignant"} for v in normalized):
        diagnosis_col = col

if id_col is None or diagnosis_col is None:
    raise ValueError(
        "Could not identify image-ID and diagnosis columns. "
        f"CSV columns: {labels_raw.columns}. Inspect labels_raw.head() and adjust the column selection."
    )

labels = (
    labels_raw
    .select(
        pl.col(id_col).cast(pl.String).str.strip_chars().alias("image_id"),
        pl.col(diagnosis_col).cast(pl.String).str.strip_chars().str.to_lowercase().alias("diagnosis")
    )
    .filter(
        pl.col("image_id").str.starts_with("ISIC_")
        & pl.col("diagnosis").is_in(["benign", "malignant"])
    )
    .unique(subset=["image_id"])
    .with_columns(
        pl.when(pl.col("diagnosis") == "malignant")
        .then(1)
        .otherwise(0)
        .cast(pl.Int8)
        .alias("label")
    )
)

print("Clean labels:", labels.shape)
display(labels.head())
print(labels.group_by("diagnosis").len().sort("diagnosis"))

## 5. Match each image with its mask and label

The filename convention is `ISIC_<id>.jpg` and `ISIC_<id>_Segmentation.png`. We join on the image ID, not on row order.

In [ ]:
image_map = {p.stem: str(p) for p in image_paths}
mask_map = {
    p.stem.replace("_Segmentation", ""): str(p)
    for p in mask_paths
}

image_records = [
    {"image_id": image_id, "image_path": path}
    for image_id, path in image_map.items()
]
mask_records = [
    {"image_id": image_id, "mask_path": path}
    for image_id, path in mask_map.items()
]

images_df = pl.DataFrame(image_records)
masks_df = pl.DataFrame(mask_records)

metadata = (
    labels
    .join(images_df, on="image_id", how="inner")
    .join(masks_df, on="image_id", how="inner")
    .select("image_id", "image_path", "mask_path", "diagnosis", "label")
    .sort("image_id")
)

print("Matched image-mask-label records:", metadata.height)
print("Unmatched labels:", labels.height - metadata.height)
display(metadata.head())

In [ ]:
# Validate paths and mask/image dimensions for every matched record.
missing_images = metadata.filter(~pl.col("image_path").map_elements(lambda x: Path(x).is_file(), return_dtype=pl.Boolean))
missing_masks = metadata.filter(~pl.col("mask_path").map_elements(lambda x: Path(x).is_file(), return_dtype=pl.Boolean))

assert missing_images.height == 0, "Some image files are missing."
assert missing_masks.height == 0, "Some mask files are missing."

dimension_errors = []
for row in metadata.iter_rows(named=True):
    with Image.open(row["image_path"]) as im, Image.open(row["mask_path"]) as mk:
        if im.size != mk.size:
            dimension_errors.append((row["image_id"], im.size, mk.size))

print("Dimension mismatches:", len(dimension_errors))
if dimension_errors:
    print(dimension_errors[:5])
    raise ValueError("Some image/mask dimensions do not match.")

assert metadata.height > 0, "No matched records found."
metadata.write_csv(TABLE_DIR / "metadata.csv")
print("Saved:", TABLE_DIR / "metadata.csv")

## 6. Exploratory data analysis

Inspect class balance, image dimensions, and a few paired image/mask examples before training.

In [ ]:
class_counts = metadata.group_by("diagnosis").len().sort("diagnosis")
display(class_counts)

plt.figure(figsize=(5, 4))
plt.bar(class_counts["diagnosis"].to_list(), class_counts["len"].to_list())
plt.title("ISIC 2016 Task 3B class distribution")
plt.xlabel("Diagnosis")
plt.ylabel("Number of images")
plt.show()

In [ ]:
def show_sample(row):
    image = Image.open(row["image_path"]).convert("RGB")
    mask = Image.open(row["mask_path"]).convert("L")

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(image)
    axes[0].set_title(f"Image: {row['image_id']}")

    axes[1].imshow(mask, cmap="gray", vmin=0, vmax=255)
    axes[1].set_title("Expert mask")

    axes[2].imshow(image)
    axes[2].imshow(mask, cmap="jet", alpha=0.35)
    axes[2].set_title(f"Overlay — {row['diagnosis']}")

    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

sample_rows = metadata.sample(n=min(3, metadata.height), seed=SEED).to_dicts()
for row in sample_rows:
    show_sample(row)

## 7. Create stratified train/validation/test splits

This challenge provides training ground truth but does not provide a public validation split for this notebook's local training workflow. We create reproducible stratified splits from the 900 labelled training records.

**Important:** Keep the test split untouched until final evaluation. These are internal held-out records, not the original challenge's hidden test set.

In [ ]:
from sklearn.model_selection import train_test_split

meta_pd = metadata.to_pandas()

train_df, temporary_df = train_test_split(
    meta_pd,
    test_size=0.30,
    stratify=meta_pd["label"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    stratify=temporary_df["label"],
    random_state=SEED
)

for name, frame in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(f"{name}: {len(frame)} records")
    print(frame["diagnosis"].value_counts().to_dict())

# Persist split membership for reproducibility.
for name, frame in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    pl.from_pandas(frame).write_csv(TABLE_DIR / f"{name}_metadata.csv")

## 8. Configure a memory-conscious PyTorch pipeline

Images are loaded from disk on demand. We resize to 256×256, use batch size 4, and set `num_workers=0` for Colab stability. Image and mask receive the same geometric augmentation.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_SIZE = 256
BATCH_SIZE = 4
NUM_WORKERS = 0
PIN_MEMORY = DEVICE.type == "cuda"

print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
class ISICDataset(Dataset):
    def __init__(self, dataframe, image_size=256, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.image_size = image_size
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        image = Image.open(row["image_path"]).convert("RGB")
        mask = Image.open(row["mask_path"]).convert("L")

        image = image.resize((self.image_size, self.image_size), Image.Resampling.BILINEAR)
        mask = mask.resize((self.image_size, self.image_size), Image.Resampling.NEAREST)

        image = np.asarray(image, dtype=np.float32) / 255.0
        mask = np.asarray(mask, dtype=np.uint8)
        mask = (mask >= 128).astype(np.float32)

        # Paired flips: the exact same transform is applied to image and mask.
        if self.augment and random.random() < 0.5:
            image = np.fliplr(image).copy()
            mask = np.fliplr(mask).copy()
        if self.augment and random.random() < 0.5:
            image = np.flipud(image).copy()
            mask = np.flipud(mask).copy()

        image = torch.from_numpy(image).permute(2, 0, 1).float()
        mask = torch.from_numpy(mask[None, ...]).float()
        label = torch.tensor(int(row["label"]), dtype=torch.long)

        # ImageNet normalization for the pretrained ResNet classifier.
        mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
        std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
        image = (image - mean) / std

        return image, mask, label

train_ds = ISICDataset(train_df, IMAGE_SIZE, augment=True)
val_ds = ISICDataset(val_df, IMAGE_SIZE, augment=False)
test_ds = ISICDataset(test_df, IMAGE_SIZE, augment=False)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)

batch = next(iter(train_loader))
print("Image batch:", batch[0].shape)
print("Mask batch:", batch[1].shape)
print("Label batch:", batch[2].shape)

## 9. Classification baseline — ResNet18

Use a pretrained ResNet18 with a two-class output. The training loop saves the best validation-loss checkpoint.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

weights = models.ResNet18_Weights.DEFAULT
classifier = models.resnet18(weights=weights)
classifier.fc = nn.Linear(classifier.fc.in_features, 2)
classifier = classifier.to(DEVICE)

classification_loss = nn.CrossEntropyLoss()
classifier_optimizer = torch.optim.AdamW(classifier.parameters(), lr=1e-4, weight_decay=1e-4)

def run_classifier_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, all_y, all_prob = 0.0, [], []

    for images, masks, labels_batch in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels_batch = labels_batch.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = classification_loss(logits, labels_batch)

            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        probs = torch.softmax(logits.detach(), dim=1)[:, 1]
        all_y.extend(labels_batch.detach().cpu().numpy().tolist())
        all_prob.extend(probs.cpu().numpy().tolist())

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, np.asarray(all_y), np.asarray(all_prob)

EPOCHS_CLS = 8
best_val_loss = float("inf")
classifier_history = []

for epoch in range(EPOCHS_CLS):
    train_loss, _, _ = run_classifier_epoch(classifier, train_loader, classifier_optimizer)
    val_loss, y_val, p_val = run_classifier_epoch(classifier, val_loader)

    val_pred = (p_val >= 0.5).astype(int)
    val_acc = accuracy_score(y_val, val_pred)
    classifier_history.append((train_loss, val_loss, val_acc))

    print(
        f"Epoch {epoch+1}/{EPOCHS_CLS} | "
        f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
        f"val_accuracy={val_acc:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(classifier.state_dict(), MODEL_DIR / "resnet18_isic_best.pth")

print("Best classifier checkpoint:", MODEL_DIR / "resnet18_isic_best.pth")

## 10. Evaluate classification on the held-out split

The test split is evaluated only after model selection using validation loss.

In [ ]:
classifier.load_state_dict(torch.load(MODEL_DIR / "resnet18_isic_best.pth", map_location=DEVICE))
test_loss, y_test, p_test = run_classifier_epoch(classifier, test_loader)
pred_test = (p_test >= 0.5).astype(int)

print("Test loss:", round(test_loss, 4))
print("Accuracy:", round(accuracy_score(y_test, pred_test), 4))
print("Precision:", round(precision_score(y_test, pred_test, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, pred_test, zero_division=0), 4))
print("F1:", round(f1_score(y_test, pred_test, zero_division=0), 4))

if len(np.unique(y_test)) == 2:
    print("ROC-AUC:", round(roc_auc_score(y_test, p_test), 4))

print(classification_report(y_test, pred_test, target_names=["benign", "malignant"], zero_division=0))

cm = confusion_matrix(y_test, pred_test)
plt.figure(figsize=(4, 4))
plt.imshow(cm, cmap="Blues")
plt.title("Classification confusion matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks([0, 1], ["Benign", "Malignant"])
plt.yticks([0, 1], ["Benign", "Malignant"])
for (i, j), value in np.ndenumerate(cm):
    plt.text(j, i, str(value), ha="center", va="center")
plt.colorbar()
plt.show()

## 11. Segmentation baseline — compact U-Net

This U-Net is intentionally small for limited Colab memory. It predicts a one-channel lesion probability map. The target masks are binary.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class SmallUNet(nn.Module):
    def __init__(self, base=16):
        super().__init__()
        self.enc1 = ConvBlock(3, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.enc3 = ConvBlock(base * 2, base * 4)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(base * 4, base * 8)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = ConvBlock(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)
        self.out = nn.Conv2d(base, 1, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return self.out(d1)

segmenter = SmallUNet(base=16).to(DEVICE)
seg_optimizer = torch.optim.AdamW(segmenter.parameters(), lr=1e-3, weight_decay=1e-5)

In [ ]:
def dice_loss_from_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    dims = (1, 2, 3)
    intersection = (probs * targets).sum(dims)
    denominator = probs.sum(dims) + targets.sum(dims)
    dice = (2 * intersection + eps) / (denominator + eps)
    return 1 - dice.mean()

def segmentation_loss(logits, targets):
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    return 0.5 * bce + 0.5 * dice_loss_from_logits(logits, targets)

@torch.no_grad()
def evaluate_segmenter(model, loader):
    model.eval()
    total_loss, dice_scores, iou_scores = 0.0, [], []

    for images, masks, labels_batch in loader:
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        logits = model(images)
        loss = segmentation_loss(logits, masks)
        total_loss += loss.item() * images.size(0)

        pred = (torch.sigmoid(logits) >= 0.5).float()
        dims = (1, 2, 3)
        intersection = (pred * masks).sum(dims)
        union = ((pred + masks) > 0).float().sum(dims)
        denom = pred.sum(dims) + masks.sum(dims)

        dice = (2 * intersection + 1e-6) / (denom + 1e-6)
        iou = (intersection + 1e-6) / (union + 1e-6)

        dice_scores.extend(dice.cpu().numpy().tolist())
        iou_scores.extend(iou.cpu().numpy().tolist())

    return (
        total_loss / len(loader.dataset),
        float(np.mean(dice_scores)),
        float(np.mean(iou_scores))
    )

def train_segmenter_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0

    for images, masks, labels_batch in loader:
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        logits = model(images)
        loss = segmentation_loss(logits, masks)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    return total_loss / len(loader.dataset)

In [ ]:
EPOCHS_SEG = 10
best_seg_val_loss = float("inf")
seg_history = []

for epoch in range(EPOCHS_SEG):
    train_loss = train_segmenter_epoch(segmenter, train_loader, seg_optimizer)
    val_loss, val_dice, val_iou = evaluate_segmenter(segmenter, val_loader)

    seg_history.append((train_loss, val_loss, val_dice, val_iou))

    print(
        f"Epoch {epoch+1}/{EPOCHS_SEG} | "
        f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
        f"val_dice={val_dice:.4f} | val_iou={val_iou:.4f}"
    )

    if val_loss < best_seg_val_loss:
        best_seg_val_loss = val_loss
        torch.save(segmenter.state_dict(), MODEL_DIR / "unet_isic_best.pth")

print("Best segmentation checkpoint:", MODEL_DIR / "unet_isic_best.pth")

## 12. Evaluate segmentation on the held-out split

In [ ]:
segmenter.load_state_dict(torch.load(MODEL_DIR / "unet_isic_best.pth", map_location=DEVICE))
test_seg_loss, test_dice, test_iou = evaluate_segmenter(segmenter, test_loader)

print("Test segmentation loss:", round(test_seg_loss, 4))
print("Test Dice:", round(test_dice, 4))
print("Test IoU:", round(test_iou, 4))

## 13. Visualize model predictions

In [ ]:
segmenter.eval()
classifier.eval()

images, masks, labels_batch = next(iter(test_loader))
images_device = images.to(DEVICE)

with torch.no_grad():
    predicted_masks = (torch.sigmoid(segmenter(images_device)) >= 0.5).cpu().numpy()
    class_probabilities = torch.softmax(classifier(images_device), dim=1)[:, 1].cpu().numpy()

fig, axes = plt.subplots(min(4, len(images)), 3, figsize=(10, 4 * min(4, len(images))))
if len(images) == 1:
    axes = np.expand_dims(axes, axis=0)

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

for i in range(min(4, len(images))):
    img = images[i].permute(1, 2, 0).numpy() * std + mean
    img = np.clip(img, 0, 1)
    true_mask = masks[i, 0].numpy()
    pred_mask = predicted_masks[i, 0]

    axes[i, 0].imshow(img)
    axes[i, 0].set_title("Input image")

    axes[i, 1].imshow(true_mask, cmap="gray")
    axes[i, 1].set_title("Ground-truth mask")

    axes[i, 2].imshow(img)
    axes[i, 2].imshow(pred_mask, cmap="jet", alpha=0.35)
    axes[i, 2].set_title(f"Predicted mask | P(malignant)={class_probabilities[i]:.2f}")

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
plt.show()

## 14. Save artifacts and metadata

Save model weights, split metadata, and evaluation summaries. Colab's `/content` storage is temporary; mount Google Drive to preserve outputs.

In [ ]:
import json

summary = {
    "dataset": "ISIC 2016 Task 3B",
    "matched_records": int(metadata.height),
    "train_records": int(len(train_df)),
    "validation_records": int(len(val_df)),
    "test_records": int(len(test_df)),
    "classification_test_loss": float(test_loss),
    "classification_test_accuracy": float(accuracy_score(y_test, pred_test)),
    "classification_test_f1": float(f1_score(y_test, pred_test, zero_division=0)),
    "segmentation_test_loss": float(test_seg_loss),
    "segmentation_test_dice": float(test_dice),
    "segmentation_test_iou": float(test_iou),
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "seed": SEED
}

with open(OUTPUT_DIR / "experiment_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

In [ ]:
# Optional: mount Google Drive and copy the project outputs.
from google.colab import drive
drive.mount("/content/drive")

DRIVE_OUTPUT = Path("/content/drive/MyDrive/AI_Assisted_Medical_Imaging")
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

shutil.copy2(MODEL_DIR / "resnet18_isic_best.pth", DRIVE_OUTPUT / "resnet18_isic_best.pth")
shutil.copy2(MODEL_DIR / "unet_isic_best.pth", DRIVE_OUTPUT / "unet_isic_best.pth")
shutil.copy2(OUTPUT_DIR / "experiment_summary.json", DRIVE_OUTPUT / "experiment_summary.json")
shutil.copy2(TABLE_DIR / "metadata.csv", DRIVE_OUTPUT / "metadata.csv")

print("Model weights, experiment summary, and metadata copied to Google Drive.")

## 15. References and limitations

- ISIC 2016 Task 3B: https://challenge.isic-archive.com/landing/2016/41/
- ISIC challenge data overview: https://challenge.isic-archive.com/data/
- Gutman et al., *Skin Lesion Analysis toward Melanoma Detection: A Challenge at the International Symposium on Biomedical Imaging (ISBI) 2016*, arXiv: https://arxiv.org/abs/1605.01397

**Limitations**
- Only 900 labelled training examples are used here.
- A random internal test split is not equivalent to an external clinical validation cohort.
- Class imbalance, acquisition differences, and skin-tone representation can affect generalization.
- The baseline models and outputs are for research and education only, not diagnosis or treatment decisions.